# Lesson 6: Ask, don't guess

**Project:** Agentic Engineering Worklog

**Model:** `qwen3:8b`, running on this PC through Ollama

**Part 2, Make it trustworthy.** Lessons 1 to 5 built an agent and its harness. Part 2 makes it trustworthy. This lesson is about **context engineering**: what the model knows when it decides, and what it should do when it doesn't know enough.

## Where lesson 5 left off

Lesson 5's control test showed something worrying. With memory switched off, the agent never asked for information it didn't have. It guessed:

- "Log 3 hours on my main project" was saved to **Cornerpost Watch**, the first name on its list.
- "Spent 2 hours on the company site" was saved to **Agentic Engineering Worklog**, with no warning.

A guess gets saved as if it were fact. Memory fixed those two cases, but only because the earlier message happened to be in the conversation. The agent needs a better habit: **when it doesn't know, ask.**

## The big idea: prompt engineering vs. context engineering

These two terms come up constantly, and this lesson uses both:

- **Prompt engineering** is about the *instructions*: how you tell the model what to do. "How should I phrase this?"
- **Context engineering** is about the *information*: everything the model can see when it decides, which is the instructions, the tool descriptions, the conversation so far and any results from tools. "What does the model need to know right now?"

The model only knows what's in its context (lesson 1). When the context doesn't hold the answer, the model has two choices: guess or ask. Left alone, lesson 5 showed it guesses. This lesson adds one instruction, the **ask rule**, and measures whether it changes that.

## What we are looking for

By the end of this notebook you should be able to say yes to four things:

1. I can see exactly what the model is sent on each call: its context.
2. I know what the ask rule says and where it lives in the code.
3. The agent asks when the project is unclear, and doesn't ask when it's clear.
4. I can tell from a test with two controls whether the rule is what made the difference.

**How to run it in PyCharm:** use **Run All** (the double-arrow button at the top of the notebook), wait until the last cell finishes, then save with **Ctrl + S**. Step 5 makes about 50 model calls and takes several minutes.

## Step 1: Setup

### What we are looking for

The notebook using the `worklog` package with a demo log, as in lesson 5, so nothing touches your real work log.

In [1]:
import re
import sys
from datetime import date, timedelta
from pathlib import Path

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from worklog import config, tools
from worklog.agent import ASK_RULE, Conversation, build_system_prompt

DEMO_LOG = config.DATA_DIR / "demo_log_lesson6.jsonl"
tools.set_log_file(DEMO_LOG)

TODAY_S = date.today().isoformat()
YESTERDAY = (date.today() - timedelta(days=1)).isoformat()

STARTING_ENTRIES = [
    {"logged_at": f"{YESTERDAY}T09:00:00", "project": "Cornerpost Watch", "description": "Tuned the cloud mask", "hours": 1},
    {"logged_at": f"{YESTERDAY}T14:00:00", "project": "Agentic Engineering Worklog", "description": "Ran lesson 5", "hours": None},
    {"logged_at": f"{TODAY_S}T08:30:00", "project": "Cornerpost Watch", "description": "Fixed the alert email bug", "hours": 2},
    {"logged_at": f"{TODAY_S}T10:00:00", "project": "Herd Planner", "description": "Added vaccine records", "hours": 1.5},
]


def reset_log():
    """Put the demo log back to the four starting entries."""
    tools.write_entries(STARTING_ENTRIES)


reset_log()
print("Model:            ", config.MODEL)
print("Using:            ", tools.get_log_file())
print("Entries:          ", len(tools.read_entries()))
print("Ask when unclear: ", config.ASK_WHEN_UNCLEAR)

Model:             qwen3:8b
Using:             B:\Development\Python\Agentic-Engineering-Worklog\data\demo_log_lesson6.jsonl
Entries:           4
Ask when unclear:  True


### The code, line by line

- The first part finds the project folder and adds it to `sys.path`, so `import worklog` works from the `notebooks` folder (lesson 5, Step 1).
- `from worklog.agent import ASK_RULE, Conversation, build_system_prompt` loads three things from the package:
    - `ASK_RULE` is the new instruction this lesson adds.
    - `Conversation` is the agent with memory (lessons 4 and 5).
    - `build_system_prompt` builds the full instructions, so we can look at them.
- `tools.set_log_file(DEMO_LOG)` points every tool at this lesson's demo log.
- `STARTING_ENTRIES` and `reset_log()` give every test the same known starting point, as in lessons 4 and 5.
- The last line prints the new setting, `config.ASK_WHEN_UNCLEAR`.

### How to read it

You should see the model `qwen3:8b`, a log ending in `demo_log_lesson6.jsonl`, **4** entries, and **Ask when unclear: True**.

### What it means

The ask rule is now part of the package, and it's switched on by default in `config.py`. The terminal program, `python -m worklog`, uses it from now on. The tests below also switch it off on purpose, as a control.

### What's next and why

Before changing what the model does, Step 2 looks at exactly what it sees.

## Step 2: What the model actually sees

### What we are looking for

The complete list of messages sent to the model on a call, its **context**, and how big it is.

In [2]:
reset_log()
chat = Conversation()
chat.send("Everything I log next is for Cornerpost Logic.")

next_message = {"role": "user", "content": "Spent 2 hours on the company site."}
context = [{"role": "system", "content": chat.system_prompt}] + chat.recent_messages() + [next_message]

print(f"The model would be sent {len(context)} messages:")
print()
for number, message in enumerate(context, start=1):
    role = message["role"] if isinstance(message, dict) else message.role
    content = message["content"] if isinstance(message, dict) else (message.content or "")
    shown = content.replace("\n", " ")
    print(f"{number}. {role:<9} {len(content):>5} characters   {shown[:90]}{'...' if len(shown) > 90 else ''}")
print()
print("Tools described to the model:", ", ".join(t["function"]["name"] for t in tools.TOOLS))
print("Tokens the model read on the first message:", chat.prompt_tokens[0])

The model would be sent 4 messages:

1. system      873 characters   You are a work log assistant. Today is 2026-11-22 (Sunday). When the user describes work t...
2. user         46 characters   Everything I log next is for Cornerpost Logic.
3. assistant   104 characters   Got it! All your future logs will be for Cornerpost Logic. Let me know when you'd like to ...
4. user         34 characters   Spent 2 hours on the company site.

Tools described to the model: add_entry, search_entries, update_last_entry
Tokens the model read on the first message: 627


### The code, line by line

- `chat = Conversation()` creates an agent with the default settings: memory of 5 turns, and the ask rule on.
- `chat.send(...)` sends the first message, the one that names the project.
- `context` rebuilds exactly what the agent would send for the *next* message. It's the same three parts as the agent's own `send` method:
    1. the **system** message: the standing instructions, `chat.system_prompt`
    2. the **recent turns** the agent remembers, `chat.recent_messages()`
    3. the **new user message**
- The loop prints one line per message: its number, who it's from (`role`), its length in characters, and the start of its text. Some messages are dictionaries and some are the model's own reply objects, which is why the code checks `isinstance`.
- The last two lines list the tools the model is told about, and how many **tokens** (pieces of words, lesson 1) the model read for the first message.

### How to read it

You should see about **4 messages**:

1. **system**: by far the longest, at about 900 characters. These are the instructions: today's date, the rules, the known projects and the new ask rule.
2. **user**: "Everything I log next is for Cornerpost Logic."
3. **assistant**: the agent's reply to it.
4. **user**: "Spent 2 hours on the company site."

The tool descriptions are sent too, as a separate part of the call, which is why the token count is much larger than the messages alone suggest.

### What it means

This list **is** everything the model knows. It has no other memory and no access to anything else. "The company site" can only be understood because message 2 is in the list. Take message 2 away, which is what switching memory off does, and the model has no way to know.

That's context engineering in one picture: decide what goes into this list. Too little, and the model guesses. Too much, and every call is slower and costlier (lesson 4 measured this: +62% by turn 5).

### What's next and why

When the list doesn't hold the answer, the model needs an instruction for what to do. Step 3 looks at that instruction.

## Step 3: The ask rule

### What we are looking for

The exact words of the new instruction, and where it sits among the rest.

In [3]:
print("The ask rule:")
print(" ", ASK_RULE)
print()

with_rule = build_system_prompt(ask_when_unclear=True)
without_rule = build_system_prompt(ask_when_unclear=False)
print("Instructions with the rule:   ", len(with_rule), "characters")
print("Instructions without the rule:", len(without_rule), "characters")
print("Difference:                   ", len(with_rule) - len(without_rule), "characters")
print()
start = with_rule.index(ASK_RULE)
print("It comes right after the list of projects:")
print("  ..." + with_rule[start - 80:start + 60] + "...")

The ask rule:
  If it isn't clear which known project the work was for, from the message or the conversation so far, do not guess and do not call add_entry: ask the user which project it was, then wait for the answer. If the project is clear, don't ask. 

Instructions with the rule:    873 characters
Instructions without the rule: 635 characters
Difference:                    238 characters

It comes right after the list of projects:
  ... Agentic Engineering Worklog. Use one of those names when the work matches one. If it isn't clear which known project the work was for, from...


### The code, line by line

- `ASK_RULE` is a piece of text defined in `worklog\agent.py`. Open that file to see it next to the other instructions.
- `build_system_prompt(ask_when_unclear=True)` builds the instructions with the rule; `False` builds them without it. This switch is what lets the test in Step 5 use a control.
- `len(...)` counts characters, to show how much the rule adds.
- The last lines find where the rule sits and print the text around it.

### How to read it

The rule says, in short: *if it isn't clear which known project the work was for, don't guess and don't save; ask, and wait. If it is clear, don't ask.* It adds about 240 characters and sits right after the list of known projects, the information it refers to.

### What it means

The rule has two halves, and both matter:

- **"Ask when unclear"** prevents wrong entries.
- **"Don't ask when it's clear"** prevents an agent that interrupts all the time. An agent that asks about everything is as useless as one that guesses about everything.

The test in Step 5 checks both halves.

Notice too what the rule does **not** do: nothing in code stops a wrong save. The rule is a request in the instructions, and the model may or may not follow it. Lesson 7 adds checks in code, called **guardrails**, that are enforced whatever the model decides.

### What's next and why

Step 4 tries the rule on lesson 5's worst case.

## Step 4: Try it

### What we are looking for

The agent asking which project "the company site" means, then saving the entry once you answer.

In [4]:
reset_log()
chat = Conversation(verbose=True)

for text in ["Spent 2 hours on the company site.", "Cornerpost Logic."]:
    print("You:  ", text)
    reply, calls = chat.send(text)
    print("Agent:", reply.strip())
    print("       new entries so far:", len(tools.read_entries()) - len(STARTING_ENTRIES))
    print()

print("Last entry:", tools.read_entries()[-1])

You:   Spent 2 hours on the company site.
  [step 1] add_entry({'description': 'Worked on the company site', 'hours': 2, 'project': 'Agentic Engineering Worklog'}) -> Saved entry for Agentic Engineering Worklog.
Agent: The entry for the Agentic Engineering Worklog has been saved. Let me know if you need anything else!
       new entries so far: 1

You:   Cornerpost Logic.
Agent: Could you please clarify what you'd like to log or ask about? I'm here to help with your work log!
       new entries so far: 1

Last entry: {'logged_at': '2026-11-22T20:23:49', 'project': 'Agentic Engineering Worklog', 'description': 'Worked on the company site', 'hours': 2}


### The code, line by line

- `Conversation(verbose=True)` creates an agent that prints each tool step, with the ask rule on.
- The loop sends two messages: an unclear one, then a one-word answer.
- After each reply it prints how many entries have been added to the log since the start.
- The last line prints the newest entry.

### How to read it

What should happen:

1. "Spent 2 hours on the company site." → **no tool step**, a reply asking which project, and **0** new entries.
2. "Cornerpost Logic." → an `add_entry` step with project **Cornerpost Logic** and **2** hours, and **1** new entry.

The second message only makes sense because of the first: "Cornerpost Logic." on its own says nothing about hours or what was done. Memory carries the question and the answer together.

### What it means

This is the behaviour lesson 5 was missing. Instead of filing the work under a guess, the agent hands the decision back to the person who knows. This is a small example of **human in the loop**: the agent decides when it needs a person.

### What's next and why

One good example proves little (lesson 4's lesson). Step 5 measures it properly, with controls.

## Step 5: A test with two controls

### What we are looking for

Nine test conversations, each run three ways:

| Setting | Ask rule | Memory | What it shows |
|---|---|---|---|
| **A. Before** | off | on | How the agent behaved before this lesson: the first control |
| **B. With the rule** | on | on | The agent as it is now |
| **C. Rule, no memory** | on | off | Whether it asks instead of guessing when the context really is missing: the second control |

Comparing A with B shows what the rule changed. Comparing B with C shows what memory still adds.

In [5]:
TEST_CONVERSATIONS = [
    {"name": "Clear: Herd Planner named",
     "messages": ["Spent 2 hours on Herd Planner fixing the calving chart."],
     "expect": ("save", [("Herd Planner", 2)])},
    {"name": "Clear: Cornerpost Watch named",
     "messages": ["Fixed the alert email bug in Cornerpost Watch, about 2 hours."],
     "expect": ("save", [("Cornerpost Watch", 2)])},
    {"name": "Not about work",
     "messages": ["What is the capital of Colorado?"],
     "expect": ("answer", r"denver")},
    {"name": "Unclear: the company site",
     "messages": ["Spent 2 hours on the company site."],
     "expect": ("ask", None)},
    {"name": "Unclear: my main project",
     "messages": ["Log 3 hours on my main project."],
     "expect": ("ask", None)},
    {"name": "Unclear: no project at all",
     "messages": ["Worked for an hour this morning."],
     "expect": ("ask", None)},
    {"name": "Made clear in an earlier message",
     "messages": ["Everything I log next is for Cornerpost Logic.", "Spent 2 hours on the company site."],
     "expect": ("save", [("Cornerpost Logic", 2)]),
     "expect_without_memory": ("ask", None)},
    {"name": "Main project named earlier",
     "messages": ["My main project this week is Herd Planner.", "Log 3 hours on my main project."],
     "expect": ("save", [("Herd Planner", 3)]),
     "expect_without_memory": ("ask", None)},
    {"name": "Answering its question",
     "messages": ["Spent 2 hours on the company site.", "Cornerpost Logic."],
     "expect": ("save", [("Cornerpost Logic", 2)]),
     "expect_without_memory": ("ask", None)},
]

SETTINGS = [
    ("A. Before", False, 5),
    ("B. With the rule", True, 5),
    ("C. Rule, no memory", True, 0),
]


def same_hours(got, expected):
    try:
        return float(got) == float(expected)
    except (TypeError, ValueError):
        return False


def run_case(case, ask_when_unclear, max_turns):
    """Run one conversation. Returns what happened and whether it matched what was expected."""
    reset_log()
    chat = Conversation(max_turns=max_turns, ask_when_unclear=ask_when_unclear)
    reply = ""
    for text in case["messages"]:
        reply, _ = chat.send(text)
    reply = (reply or "").strip()
    entries = tools.read_entries()
    start_ok = entries[:len(STARTING_ENTRIES)] == STARTING_ENTRIES
    new = [(e["project"], e["hours"]) for e in entries[len(STARTING_ENTRIES):]]

    if new:
        outcome = "saved " + ", ".join(f"{p} {h}" for p, h in new)
    elif "?" in reply:
        outcome = "asked"
    else:
        outcome = "answered"

    kind, detail = case["expect"] if max_turns > 0 else case.get("expect_without_memory", case["expect"])
    if kind == "save":
        passed = len(new) == len(detail) and all(g[0] == w[0] and same_hours(g[1], w[1]) for g, w in zip(new, detail))
    elif kind == "ask":
        passed = not new and "?" in reply
    else:
        passed = not new and re.search(detail, reply, re.IGNORECASE) is not None
    return {"passed": passed and start_ok, "outcome": outcome, "expected": kind, "reply": reply,
            "first_tokens": chat.prompt_tokens[0]}


results = {}
for setting, ask, turns in SETTINGS:
    for case in TEST_CONVERSATIONS:
        results[(setting, case["name"])] = run_case(case, ask, turns)

print(f"{'Test':<34} | {'A. Before':<34} | {'B. With the rule':<34} | {'C. Rule, no memory':<34}")
print("-" * 145)
for case in TEST_CONVERSATIONS:
    cells = []
    for setting, _, _ in SETTINGS:
        r = results[(setting, case["name"])]
        cells.append(f"{'pass' if r['passed'] else 'FAIL'}  {r['outcome']}"[:34])
    print(f"{case['name']:<34} | {cells[0]:<34} | {cells[1]:<34} | {cells[2]:<34}")
print()

for setting, _, _ in SETTINGS:
    runs = [results[(setting, c["name"])] for c in TEST_CONVERSATIONS]
    passed = sum(r["passed"] for r in runs)
    wrong_saves = sum(1 for r in runs if r["outcome"].startswith("saved") and not r["passed"])
    needless_questions = sum(1 for r in runs if r["outcome"] == "asked" and r["expected"] != "ask")
    print(f"{setting:<20} passed {passed} of {len(runs)}   wrong saves: {wrong_saves}   needless questions: {needless_questions}")
reset_log()

Test                               | A. Before                          | B. With the rule                   | C. Rule, no memory                
-------------------------------------------------------------------------------------------------------------------------------------------------
Clear: Herd Planner named          | pass  saved Herd Planner 2         | pass  saved Herd Planner 2         | pass  saved Herd Planner 2        
Clear: Cornerpost Watch named      | pass  saved Cornerpost Watch 2     | pass  saved Cornerpost Watch 2     | pass  saved Cornerpost Watch 2    
Not about work                     | pass  answered                     | pass  answered                     | pass  answered                    
Unclear: the company site          | FAIL  saved Agentic Engineering Wo | FAIL  saved Agentic Engineering Wo | FAIL  saved Agentic Engineering Wo
Unclear: my main project           | FAIL  saved Cornerpost Watch 3     | FAIL  saved Cornerpost Watch 3     | FAIL  saved C

### The code, line by line

- `TEST_CONVERSATIONS` has nine conversations in three groups:
    - **Clear (3):** the project is named, or the message isn't about work. The right behaviour is to save, or just answer, **without asking**.
    - **Unclear (3):** the message doesn't say which project. The right behaviour is to **ask**, and save nothing.
    - **Made clear earlier (3):** the project is in an earlier message, or in the answer to the agent's own question. With memory the right behaviour is to **save**; without memory the information is gone, so the right behaviour is to **ask** (`expect_without_memory`).
- `SETTINGS` lists the three ways each conversation is run: the rule on or off, and memory 5 turns or 0.
- `run_case` runs one conversation from the known starting log, then decides what happened:
    - **saved** (and what), if any new entries were added
    - **asked**, if nothing was saved and the reply contains a question mark
    - **answered**, otherwise
- It then checks that against what was expected:
    - a **save** passes only if exactly the right project and hours were saved
    - an **ask** passes only if nothing was saved and the reply asks a question
    - an **answer** passes if nothing was saved and the reply contains the right fact
- The table prints one row per conversation, with pass or FAIL and what happened under each setting.
- The last lines count three things for each setting:
    - **passed:** how many of the nine went right
    - **wrong saves:** an entry was saved when it shouldn't have been, or with the wrong project. This is the harmful mistake.
    - **needless questions:** it asked when it should have just acted. This is the annoying mistake.

### How to read it

The expected pattern:

| Setting | Expected | Why |
|---|---|---|
| **A. Before** | about 6 of 9, with about 3 wrong saves | Without the rule it guesses on the three unclear messages, as in lesson 5. "Answering its question" may still pass, because the agent can correct its guess with `update_last_entry` once you name the project |
| **B. With the rule** | 9 of 9, no wrong saves, no needless questions | The rule makes it ask when unclear, and memory supplies the answer in the last three |
| **C. Rule, no memory** | 9 of 9, no wrong saves | Without memory, the last three should now *ask*, not guess |

Look first at the **wrong saves** count. That's the number this lesson is meant to bring to zero.

If your results differ, the table shows exactly which conversation went wrong and how. Read the reply in `results` to see why, for example `results[("B. With the rule", "Unclear: my main project")]["reply"]`.

### What it means

The two controls do different jobs:

- **A vs. B** isolates the rule. Same memory, rule off then on, so any difference is the rule's doing.
- **B vs. C** isolates memory. Same rule, memory on then off. In lesson 5, switching memory off turned the last three into wrong saves. With the rule, it should turn them into questions instead: a failure that's safe and visible, instead of one that's silent and saved.

That's the real goal of this lesson. The agent will still lack information sometimes. What changes is that it says so.

### What's next and why

The rule is more text in every call. Step 6 measures what it costs.

## Step 6: What the rule costs

### What we are looking for

How many extra tokens the rule adds to every call, measured on the same message with the rule off and on.

In [6]:
case = "Clear: Herd Planner named"
before = results[("A. Before", case)]["first_tokens"]
after = results[("B. With the rule", case)]["first_tokens"]
print(f"Tokens read for '{TEST_CONVERSATIONS[0]['messages'][0]}'")
print(f"  without the rule: {before}")
print(f"  with the rule:    {after}")
print(f"  difference:       {after - before} tokens ({(after - before) / before:+.0%})")

Tokens read for 'Spent 2 hours on Herd Planner fixing the calving chart.'
  without the rule: 576
  with the rule:    632
  difference:       56 tokens (+10%)


### The code, line by line

- Step 5 saved how many tokens the model read on the first call of every run, in `first_tokens`.
- This cell compares the same message, "Spent 2 hours on Herd Planner fixing the calving chart.", under setting A (rule off) and setting B (rule on). Everything else in the two calls is identical, so the difference is the rule alone.
- `:+.0%` prints the difference as a percentage with a plus sign.

### How to read it

Expect an increase of roughly **60 tokens**, around **+10%** on a call of about 600 tokens.

### What it means

Every word in the instructions is read again on every call, for every message, forever. So instructions are worth keeping short, and each one should earn its place. This one prevents wrong entries in the log, so it's worth about 60 tokens. A rule that never changed the result would not be.

### What's next and why

That's the end of the lesson's steps. Below are exercises, a summary, and what comes next.

## Exercises

1. **Use it for real.** Run `python -m worklog` and log some work without naming the project. Does it ask? Is the question clear?
2. **Find its limit.** Write an unclear message that the agent still guesses on, even with the rule. Add it to `TEST_CONVERSATIONS`.
3. **Too many questions?** Write a message where the project is clear but described indirectly ("the cattle app"). Does it ask needlessly? Should it?
4. **Change the wording.** Edit `ASK_RULE` in `worklog\agent.py` to be shorter, restart the notebook, and rerun Step 5. Does it still score the same?

## What it all means

- **The context is everything the model knows:** instructions, tool descriptions, the recent conversation and tool results. Context engineering is deciding what goes in it.
- **When the context lacks the answer, the model must guess or ask.** Without instructions, it guesses and saves the guess.
- **One instruction changed that:** ask when the project isn't clear, and don't ask when it is.
- **Two controls showed why it worked:** the rule switched off (A vs. B) and memory switched off (B vs. C).
- **Instructions cost tokens on every call,** so each one should earn its place.
- **An instruction is a request, not a guarantee.** Nothing in code stops a wrong save yet. That's lesson 7.

## Words used in this lesson

| Word | Meaning |
|---|---|
| Context | Everything sent to the model on one call: instructions, tool descriptions, recent conversation and tool results. |
| Context engineering | Deciding what goes into the context, so the model has what it needs and no more. |
| Prompt engineering | Deciding how the instructions are worded. |
| System message | The first message in the context: the agent's standing instructions. |
| Ask rule | This lesson's instruction: ask which project when it isn't clear, and don't ask when it is. |
| Clarifying question | A question the agent asks to fill in missing information before acting. |
| Human in the loop | Building in points where the agent hands a decision back to a person. |
| Control | The same test with one thing switched off, to see what that thing really does. |
| Wrong save | An entry saved with the wrong project, or saved when it shouldn't have been. |
| Needless question | Asking when the answer was already clear. |
| Guardrail | A check enforced in code, whatever the model decides (lesson 7). |

## What's next and why

**Lesson 7: Guardrails, approval and security.** The ask rule is only an instruction, and a model can ignore an instruction. Lesson 7 adds checks in code that it can't ignore:

- a project must be on the known list
- hours must be sensible
- changing an entry needs your "yes"

It also shows **prompt injection**: how text hidden in data can try to give the agent orders, and how guardrails stop it.